In [25]:
import sys
BASE_DIR = "../../.."
sys.path.insert(0, BASE_DIR)

BASE_DIR = "../../../../alexander_workspace/exp"
sys.path.insert(1, BASE_DIR)

import pandas as pd
import numpy as np
import ast
import random
import json
from time import time
import gc
import os
import joblib
import chromadb
from tqdm import tqdm
from dataclasses import dataclass, field
from sentence_transformers import SentenceTransformer
from typing import Dict, List
from dataclasses import dataclass
import pyarrow as pa

random.seed(42)

from src.agents.hosted import CustomAgent
from src.utils import ReaderMetrics

from embedder.ChromaConnector import (ChromaConnection, 
                                      VectorDBConnectionConfig, 
                                      VectorDBInstance)

CONTEXTS_DATASET_PATH = "../../../data/squadv2/contexts.csv"
QA_DATASET_PATH = "../../../data/squadv2/qa_dataset.csv"
AGENT_MODEL_PATH = "../../../models/Qwen/Qwen2.5-7B-Instruct" # "../../../models/Undi95/Meta-Llama-3-8B-Instruct-hf" | "../../../models/Qwen/Qwen2.5-7B-Instruct"

In [2]:
PARAMS = {
    'version': "3.2",
    'num_samples': 2000,
    'num_contexts': 15,
    'model': AGENT_MODEL_PATH,
    'system_prompt': "You are an AI assistant who helps solve user issues.",
    "item_format": "- [{score}] {document}",
    "user_prompt": 'Answer the question using the available information from the texts in the list below. Each text has a corresponding real-value score of its relevance to the question in square brackets at the beginning. Scores are ranged from 0.0 (the text is not suitable for generating an answer based on it) to 1.0 (the text is suitable for generating an answer based on it). Use this information. Choose texts with high enough relevance scores. If, based on the specified scores, there are no texts in the list that are relevant enough to generate answer based on them, then generate the following answer: "I do not have an answer to your question". Generate answer only in English. Do not duplicate the question in the answer. Generate only the answer to the specified question. Answer need to be short. Do not generate anything extra.',
    "prompt_format": "{user_p}\n\nAvailable information:\n{cnt_list}\n\nQuestion:\n{q}\n\nAnswer:\n",
    'gen_strat': {'max_new_tokens': 1024},
    'stub_answer': "I do not have an answer to your question",
    "scores_dataset_path": "/home/jovyan/work/alexander_workspace/exp/cosine_scores_squad2.csv",
    'vdb_context_info': {'path': "/home/jovyan/work/RAG-project-SMILES-2024-/data/squadv2/chroma/dbs/v3", 'db': {'db': 'squadv2', 'table': 'contexts'}},
    'vdb_query_info': {'path': "/home/jovyan/work/RAG-project-SMILES-2024-/data/squadv2/chroma/dbs/v3", 'db': {'db': 'squadv2', 'table': 'questions'}}
}

METADATA_SAVE_NAME = 'metadata.json'
USER_PROPMTS_SAVE_NAME = 'user_prompts.json'
PARAMS_SAVE_NAME = 'hyperp.json'
GEN_ANSW_SAVE_NAME = 'generation_info.json'
SCORES_SAVE_NAME = 'scores.json'
LOGS_SAVE_DIR = './logs'
META_INFO_DIR_NAME = 'gen_metainfo'

if os.path.exists(f'{LOGS_SAVE_DIR}/v{PARAMS["version"]}'):
    print("Dir exists")
else:
    print("Creating Dir...")
    os.mkdir(f'{LOGS_SAVE_DIR}/v{PARAMS["version"]}')
    os.mkdir(f'{LOGS_SAVE_DIR}/v{PARAMS["version"]}/{META_INFO_DIR_NAME}')

Creating Dir...


### Подключение к агенту

In [3]:
agent = CustomAgent(PARAMS['model'], output_logits=False, use_cache=True, output_attentions=False, output_scores=False, output_hidden_states=False)
output = agent.generate(user_prompt="what is wrong with humanity?", system_prompt=PARAMS['system_prompt'], gen_strategy=PARAMS['gen_strat'])
print(output[0])

Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

The question of what is "wrong" with humanity is complex and multifaceted, as it can be interpreted in various ways. Here are a few perspectives:

1. **Conflict and Violence**: Throughout history, humans have engaged in wars, conflicts, and violence, often over resources, power, or beliefs.

2. **Environmental Degradation**: Many people believe that humanity's actions have led to significant environmental damage, including climate change, deforestation, pollution, and loss of biodiversity.

3. **Inequality**: There are persistent issues of inequality based on race, gender, socioeconomic status, and other factors, which lead to disparities in access to resources, opportunities, and justice.

4. **Moral Ambiguity**: Humans sometimes struggle with moral dilemmas and ethical behavior, leading to actions that harm others or the environment.

5. **Short-term Thinking**: Some argue that human societies often prioritize short-term gains over long-term sustainability and well-being.

6. **Psych

### Формируем список контекстов для каждого запроса со скорами

In [4]:
vdb_contexts_conf = VectorDBConnectionConfig(path=PARAMS['vdb_context_info']['path'], db_info=PARAMS['vdb_context_info']['db'])
connector_contexts = ChromaConnection(config=vdb_contexts_conf)
connector_contexts.count_items()

19029

In [5]:
vdb_query_conf = VectorDBConnectionConfig(path=PARAMS['vdb_query_info']['path'], db_info=PARAMS['vdb_query_info']['db'])
connector_query = ChromaConnection(config=vdb_query_conf)
print(connector_query.count_items())

86821


In [6]:
scores_df = pd.read_csv(PARAMS["scores_dataset_path"])
dataset_df = pd.read_csv(QA_DATASET_PATH)

In [7]:
scores_df

,query_id,contexts_ids,cos_dists
0,id0,"['id10', 'id0', 'id45', 'id53', 'id2', 'id26',...","[0.15416234731674194, 0.15581238269805908, 0.1..."
1,id1,"['id4', 'id0', 'id5', 'id3', 'id64', 'id50', '...","[0.15261220932006836, 0.16854727268218994, 0.1..."
2,id2,"['id10', 'id0', 'id1', 'id53', 'id12', 'id23',...","[0.13007789850234985, 0.13846325874328613, 0.1..."
3,id3,"['id8', 'id6114', 'id19', 'id6115', 'id6', 'id...","[0.16860461235046387, 0.17995333671569824, 0.1..."
4,id4,"['id84', 'id67', 'id13616', 'id94', 'id66', 'i...","[0.16440743207931519, 0.17258894443511963, 0.1..."
...,...,...,...
86816,id86816,"['id18973', 'id18917', 'id18932', 'id18934', '...","[0.11931037902832031, 0.1748185157775879, 0.18..."
86817,id86817,"['id3367', 'id3362', 'id3360', 'id3358', 'id16...","[0.18268996477127075, 0.18880540132522583, 0.1..."
86818,id86818,"['id18973', 'id18916', 'id18927', 'id18923', '...","[0.13600891828536987, 0.14367049932479858, 0.1..."
86819,id86819,"['id18973', 'id18917', 'id18934', 'id18924', '...","[0.1315150260925293, 0.16991901397705078, 0.17..."


In [8]:
dataset_df

,question,answer,relevant_context_id,metadata
0,When did Beyonce start becoming popular?,in the late 1990s,0,{'base_id': '56be85543aeaaa14008c9063'}
1,What areas did Beyonce compete in when she was...,singing and dancing,0,{'base_id': '56be85543aeaaa14008c9065'}
2,When did Beyonce leave Destiny's Child and bec...,2003,0,{'base_id': '56be85543aeaaa14008c9066'}
3,In what city and state did Beyonce grow up?,"Houston, Texas",0,{'base_id': '56bf6b0f3aeaaa14008c9601'}
4,In which decade did Beyonce become famous?,late 1990s,0,{'base_id': '56bf6b0f3aeaaa14008c9602'}
...,...,...,...,...
86816,In what US state did Kathmandu first establish...,Oregon,18973,{'base_id': '5735d259012e2f140011a09d'}
86817,What was Yangon previously known as?,Rangoon,18973,{'base_id': '5735d259012e2f140011a09e'}
86818,With what Belorussian city does Kathmandu have...,Minsk,18973,{'base_id': '5735d259012e2f140011a09f'}
86819,In what year did Kathmandu create its initial ...,1975,18973,{'base_id': '5735d259012e2f140011a0a0'}


In [9]:
CONTEXTS_LIST_IDS = []
for i in tqdm(range(PARAMS['num_samples'])):
    cur_scores = ast.literal_eval(scores_df['cos_dists'][i])
    cur_contexts = ast.literal_eval(scores_df['contexts_ids'][i])

    cur_list_ids = [(round(1-score, 5), cntx) for score, cntx in zip(cur_scores, cur_contexts)]
    
    CONTEXTS_LIST_IDS.append(cur_list_ids[:PARAMS['num_contexts']])

100%|██████████| 2000/2000 [00:00<00:00, 14742.88it/s]


In [10]:
CONTEXTS_LIST_IDS[0]

[(0.84584, 'id10'),
 (0.84419, 'id0'),
 (0.84188, 'id45'),
 (0.83928, 'id53'),
 (0.83849, 'id2'),
 (0.83739, 'id26'),
 (0.83638, 'id12'),
 (0.83603, 'id33'),
 (0.83575, 'id52'),
 (0.83561, 'id50'),
 (0.83555, 'id56'),
 (0.83433, 'id44'),
 (0.833, 'id11'),
 (0.83265, 'id47'),
 (0.82711, 'id14')]

### Готовим промпт

In [11]:
USER_PROMPTS = []
gc.collect()
for i in tqdm(range(len(CONTEXTS_LIST_IDS))):
    cur_question = connector_query.read([scores_df['query_id'][i]],includes=['documents'])[0].document
    documents_list = []
    for j in range(len(CONTEXTS_LIST_IDS[i])):
        cur_doc = connector_contexts.read([CONTEXTS_LIST_IDS[i][j][1]],includes=['documents'])[0].document
        cur_score = CONTEXTS_LIST_IDS[i][j][0]

        documents_list.append(PARAMS['item_format'].format(score=cur_score, document=cur_doc.strip()))

    documents_list = '\n'.join(documents_list)
    USER_PROMPTS.append(PARAMS['prompt_format'].format(user_p=PARAMS['user_prompt'], cnt_list=documents_list, q=cur_question))

100%|██████████| 2000/2000 [00:10<00:00, 186.07it/s]


In [12]:
with open(f"{LOGS_SAVE_DIR}/v{PARAMS['version']}/{USER_PROPMTS_SAVE_NAME}", 'w', encoding='utf-8') as fp:
    fp.write(json.dumps(USER_PROMPTS, ensure_ascii=False, indent=1))

# сохраняем конфигурацию эксперимента
with open(f"{LOGS_SAVE_DIR}/v{PARAMS['version']}/{PARAMS_SAVE_NAME}", 'w', encoding='utf-8') as fp:
    fp.write(json.dumps(PARAMS, ensure_ascii=False, indent=1))

In [13]:
print(USER_PROMPTS[0])

Answer the question using the available information from the texts in the list below. Each text has a corresponding real-value score of its relevance to the question in square brackets at the beginning. Scores are ranged from 0.0 (the text is not suitable for generating an answer based on it) to 1.0 (the text is suitable for generating an answer based on it). Use this information. Choose texts with high enough relevance scores. If, based on the specified scores, there are no texts in the list that are relevant enough to generate answer based on them, then generate the following answer: "I do not have an answer to your question". Generate answer only in English. Do not duplicate the question in the answer. Generate only the answer to the specified question. Answer need to be short. Do not generate anything extra.

Available information:
- [0.84584] Beyoncé's first solo recording was a feature on Jay Z's "'03 Bonnie & Clyde" that was released in October 2002, peaking at number four on th

In [14]:
del scores_df
gc.collect()

66

### Генерируем ответы на вопросы

In [15]:
generate_answers = []
display_iter = 100
s_time = time()
for i in tqdm(range(len(USER_PROMPTS))):
    pred_answer, meta_info = agent.generate(user_prompt=USER_PROMPTS[i], system_prompt=PARAMS['system_prompt'], gen_strategy=PARAMS['gen_strat'])
    generate_answers.append(pred_answer)

    # logits = torch.cat(metainfo['logits'], 0).cpu().detach().numpy()
    # logits_int8 = logits.astype('int8') 
    # token_logits = {f"token_{i}": token_logits for i, token_logits in enumerate(logits_int8)}
    # pa_table = pa.table(token_logits)
    # pa.parquet.write_table(pa_table, f"{LOGS_SAVE_DIR}/v{PARAMS['version']}/{META_INFO_DIR_NAME}/logits_{i}.parquet")
    
    if i % display_iter == 0:
        print(f"\n[{i}]: \nGEN: {pred_answer}\nGOLD: {dataset_df['answer'][i]}")
e_time = time()

  0%|          | 1/2000 [00:02<1:08:43,  2.06s/it]


[0]: 
GEN: Beyoncé started becoming popular after the release of her first solo album, Dangerously in Love, in 2003.
GOLD: in the late 1990s


  5%|▌         | 101/2000 [02:04<36:20,  1.15s/it]


[100]: 
GEN: I do not have an answer to your question.
GOLD: eleven


 10%|█         | 201/2000 [04:04<34:18,  1.14s/it]

...

 90%|█████████ | 1801/2000 [35:11<03:50,  1.16s/it]


[1800]: 
GEN: I do not have an answer to your question.
GOLD: IXI


 95%|█████████▌| 1901/2000 [37:08<01:43,  1.05s/it]


[1900]: 
GEN: I do not have an answer to your question.
GOLD: September 12, 2006


100%|██████████| 2000/2000 [39:02<00:00,  1.17s/it]


In [16]:
# сохраняем используемые контексты + сгнерированные ответы
gen_info = []
for i in range(PARAMS['num_samples']):
    formated_contexts = [(float(item[0]), item[1]) for item in CONTEXTS_LIST_IDS[i]]
    cur_item = {'gen_answer': str(generate_answers[i]), 'used_contexts': formated_contexts}
    gen_info.append(cur_item)

with open(f"{LOGS_SAVE_DIR}/v{PARAMS['version']}/{GEN_ANSW_SAVE_NAME}", 'w', encoding='utf-8') as fp:
    fp.write(json.dumps(gen_info, ensure_ascii=False, indent=1))

with open(f"{LOGS_SAVE_DIR}/v{PARAMS['version']}/{METADATA_SAVE_NAME}", 'w', encoding='utf-8') as fp:
    fp.write(json.dumps({'elapsed_time': e_time - s_time}, ensure_ascii=False, indent=1))

### Оцениваем качество

In [17]:
LOADING_VERSION = "3.2"

In [18]:
import nltk
nltk.download('punkt')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to /home/jovyan/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /home/jovyan/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [19]:
with open(f'{LOGS_SAVE_DIR}/v{LOADING_VERSION}/{GEN_ANSW_SAVE_NAME}','r', encoding='utf8') as fd:
    predicted_answers = list(map(lambda v: v['gen_answer'], json.loads(fd.read())))

In [20]:
metrics = ReaderMetrics(base_dir="../../..", model_path='en_electra_base')

Loading Meteor...
Loading ExactMatch


In [21]:
dataset_df = pd.read_csv(QA_DATASET_PATH)

In [22]:
target_scores = {
    'BLEU2': [], 'BLEU1': [],
    'ExactMatch': [],'METEOR': [],
    'BertScore': [],
    'Levenshtain': [],
    'ROUGEL': []}

stub_scores = {
    'BLEU2': [], 'BLEU1': [],
    'ExactMatch': [],'METEOR': [],
    'BertScore': [],
    'Levenshtain': [],
    'ROUGEL': []}

show_step = 50

process = tqdm(range(PARAMS['num_samples']))
target_answers =  dataset_df['answer'].to_list()[:PARAMS['num_samples']]
tmp_stub_pred_answers = []
for i in process:
    
    predicted_answer = predicted_answers[i]
    target_answer = target_answers[i]

    target_scores['BLEU1'] += metrics.bleu1([predicted_answer], [target_answer])
    target_scores['BLEU2'] += metrics.bleu2([predicted_answer], [target_answer])
    target_scores['ExactMatch'] += metrics.exact_match([predicted_answer], [target_answer])
    target_scores['METEOR'] += metrics.meteor([predicted_answer], [target_answer])
    target_scores['Levenshtain'] += metrics.levenshtain_score([predicted_answer], [target_answer])
    target_scores['ROUGEL'] += metrics.rougel([predicted_answer], [target_answer])

    stub_pred_answer = predicted_answer
    tmp_stub_pred_answers.append(stub_pred_answer)
    
    stub_scores['BLEU1'] += metrics.bleu1([stub_pred_answer], [PARAMS['stub_answer']])
    stub_scores['BLEU2'] += metrics.bleu2([stub_pred_answer], [PARAMS['stub_answer']])
    stub_scores['ExactMatch'] += metrics.exact_match([stub_pred_answer], [PARAMS['stub_answer']])
    stub_scores['METEOR'] += metrics.meteor([stub_pred_answer], [PARAMS['stub_answer']])
    stub_scores['Levenshtain'] += metrics.levenshtain_score([stub_pred_answer], [PARAMS['stub_answer']])
    stub_scores['ROUGEL'] += metrics.rougel([stub_pred_answer], [PARAMS['stub_answer']])
            
    if i % show_step == 0:
        process.set_postfix({m_name: np.mean(score) for m_name, score in stub_scores.items()})

target_scores = {m_name: round(float(np.mean(score)), 5) for m_name, score in target_scores.items()}
target_scores['BertScore'] = metrics.bertscore(predicted_answers, target_answers)

stub_scores = {m_name: round(float(np.mean(score)), 5) for m_name, score in stub_scores.items()}
stub_scores['BertScore'] = metrics.bertscore(tmp_stub_pred_answers, [PARAMS['stub_answer']]*len(tmp_stub_pred_answers))
stub_scores['elapsed_time_sec'] = round(float(process.format_dict["elapsed"]), 3)

/home/jovyan/work/alexander_workspace/conda/rag_alex/lib/python3.11/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/jovyan/work/alexander_workspace/conda/rag_alex/lib/python3.11/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
100%|██████████| 2000/2000 [05:43<00:00,  5.82it/s, BLEU2=0.867, BLEU1=0.874, ExactMatch=0.983, METEOR=0.972, BertScore=nan, Levenshtain=2.4, ROUGEL=0.983] 


In [23]:
LOADING_VERSION

'3.2'

In [24]:
with open(f"{LOGS_SAVE_DIR}/v{LOADING_VERSION}/{SCORES_SAVE_NAME}", 'w', encoding='utf-8') as fp:
    fp.write(json.dumps({'target_answers': target_scores, 'stub_answers': stub_scores}, ensure_ascii=False, indent=1))

### Смотрим: встречается ли релевантный контекст

In [26]:
dataset_df = pd.read_csv(QA_DATASET_PATH)
scores_df = pd.read_csv(PARAMS["scores_dataset_path"])

In [31]:
rel_cntx_ids = dataset_df['relevant_context_id'][:PARAMS['num_samples']].tolist()

In [35]:
retrieved_cntx_ids = list(map(lambda item: ast.literal_eval(item), scores_df['contexts_ids'][:PARAMS['num_samples']]))
retrieved_cntx_ids = list(map(lambda items: list(map(lambda item: int(item[2:]), items)), retrieved_cntx_ids))

In [40]:
true_cnt_exist = [true_id in retr_ids for true_id ,retr_ids in zip(rel_cntx_ids, retrieved_cntx_ids)]

In [45]:
sum(true_cnt_exist)

16